<a href="https://colab.research.google.com/github/obaidah3/rag-ecommerce-chatbot/blob/main/03_intent_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. Intent Classifier
Supervised classifier on the Bitext customer-support dataset's gold `intent` column, condensed from 27 fine-grained intents into 7 routing groups.

**Why TF-IDF + Linear SVM (LinearSVC)?**
- Gold labels already exist, so this is plain supervised text classification.
- With 26.8k short instruction-style texts, a linear SVM on TF-IDF typically matches or beats a from-scratch deep model, trains in seconds, and is trivial to explain in the assessment — a deliberate complexity trade-off given the 1-day deadline.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/chatbot_project"
MODELS_DIR = f"{PROJECT_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Models will be saved to:", MODELS_DIR)

Mounted at /content/drive
Models will be saved to: /content/drive/MyDrive/chatbot_project/models


In [ ]:
!pip install -q datasets scikit-learn joblib


In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()
print(df.shape)
df[["instruction", "intent", "category"]].head()


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

(26872, 5)


,instruction,intent,category
0,question about cancelling order {{Order Number}},cancel_order,ORDER
1,i have a question about cancelling oorder {{Or...,cancel_order,ORDER
2,i need help cancelling puchase {{Order Number}},cancel_order,ORDER
3,I need to cancel purchase {{Order Number}},cancel_order,ORDER
4,"I cannot afford this order, cancel purchase {{...",cancel_order,ORDER


In [ ]:
GROUP_MAP = {
    # order_status
    "track_order": "order_status", "delivery_options": "order_status", "delivery_period": "order_status",
    # order_management
    "cancel_order": "order_management", "change_order": "order_management", "place_order": "order_management",
    "change_shipping_address": "order_management", "set_up_shipping_address": "order_management",
    # billing_and_refunds
    "check_invoice": "billing_and_refunds", "get_invoice": "billing_and_refunds", "get_refund": "billing_and_refunds",
    "payment_issue": "billing_and_refunds", "check_refund_policy": "billing_and_refunds",
    "check_cancellation_fee": "billing_and_refunds", "check_payment_methods": "billing_and_refunds",
    "track_refund": "billing_and_refunds",
    # account_management
    "create_account": "account_management", "edit_account": "account_management",
    "delete_account": "account_management", "switch_account": "account_management",
    "recover_password": "account_management", "registration_problems": "account_management",
    # complaint
    "complaint": "complaint", "review": "complaint",
    "contact_customer_service": "complaint", "contact_human_agent": "complaint",
    # out_of_scope
    "newsletter_subscription": "out_of_scope",
}

df["intent_group"] = df["intent"].map(GROUP_MAP)
assert df["intent_group"].isna().sum() == 0, "في intents لسه ناقصة من الـ mapping!"
df["intent_group"].value_counts()

,count
intent_group,
billing_and_refunds,7939
account_management,5986
order_management,4963
complaint,3996
order_status,2989
out_of_scope,999


> **Note:** double-check the exact intent label strings against `df['intent'].unique()` for your downloaded dataset version and adjust `GROUP_MAP` — Bitext's label spelling has varied slightly across dataset versions. Run the cell below first if labels don't match.

In [ ]:
print(sorted(df['intent'].unique()))


['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["instruction"], df["intent_group"], test_size=0.15, random_state=42, stratify=df["intent_group"]
)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=30000, stop_words="english")),
    ("clf", LinearSVC(class_weight="balanced"))
])
pipe.fit(X_train, y_train)


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=30000, min_df=2,
                                 ngram_range=(1, 2), stop_words='english')),
                ('clf', LinearSVC(class_weight='balanced'))])

In [ ]:
preds = pipe.predict(X_test)
print(classification_report(y_test, preds))


                     precision    recall  f1-score   support

 account_management       1.00      1.00      1.00       898
billing_and_refunds       1.00      1.00      1.00      1191
          complaint       1.00      1.00      1.00       599
   order_management       0.99      1.00      1.00       745
       order_status       1.00      0.99      1.00       448
       out_of_scope       1.00      0.99      1.00       150

           accuracy                           1.00      4031
          macro avg       1.00      1.00      1.00      4031
       weighted avg       1.00      1.00      1.00      4031



In [ ]:
import joblib, os
joblib.dump(pipe, f"{MODELS_DIR}/intent_classifier.joblib")
print("saved")


saved


In [ ]:
pipe.predict([
    "Where is my package?",
    "I want to cancel my order",
    "This is unacceptable, I've been waiting for 3 weeks!",
    "hi there"
])


array(['order_status', 'order_management', 'billing_and_refunds',
       'billing_and_refunds'], dtype=object)

In [ ]:
print(sorted(df['intent'].unique()))

['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']
